# Atelier 3 : Apprentissage supervisé
## Arbre de décision (prédiction du diabète)

**Objectif :** construire, évaluer et optimiser un arbre de décision pour prédire si une patiente est diabétique à partir de ses caractéristiques médicales (base *Pima Indians Diabetes*).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
from IPython.display import Image
import pydotplus
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix

col_names = ['pregnant', 'glucose', 'bp', 'skin', 'insulin', 'bmi', 'pedigree', 'age', 'label']
# Le fichier Kaggle contient une ligne d'en-tête : header=0 la remplace par nos noms de colonnes
diabetes_data = pd.read_csv("C:/Users/USER/PycharmProjects/TP2/Decision_Tree/diabetes.csv", header=0, names=col_names)
diabetes_data.head()

### Question 9 : Prétraitement des données

**a) Graphes pour mieux comprendre les données**

In [ ]:
diabetes_data.info()
print(diabetes_data.describe())

# Répartition de la variable cible
sns.countplot(data=diabetes_data, x='label')
plt.title("Répartition des classes (0 = non diabétique, 1 = diabétique)")
plt.show()

# Distribution de chaque variable
diabetes_data.hist(figsize=(12, 9), bins=20)
plt.tight_layout()
plt.show()

# Corrélations entre les variables
plt.figure(figsize=(8, 6))
sns.heatmap(diabetes_data.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.show()

**Interprétation :** *(à rédiger après exécution)*

**b) Type du problème de classification**

Il s'agit d'un problème de **classification supervisée binaire** : la variable cible `label` ne prend que deux valeurs (0 = non diabétique, 1 = diabétique), et les étiquettes sont connues pour toutes les observations d'apprentissage.

**c) Variables cibles (Y) et caractéristiques (X)**

In [ ]:
feature_cols = ['pregnant', 'insulin', 'bmi', 'age', 'glucose', 'bp', 'pedigree']
X = diabetes_data[feature_cols]   # caractéristiques
y = diabetes_data.label           # variable cible

### Question 10 : Construction du modèle

**a) Découpage en ensemble d'apprentissage et ensemble de test** (70 % / 30 %)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

**b) Arbre de décision (indice de Gini)**

In [ ]:
# Create Decision Tree classifer object
clf = DecisionTreeClassifier()
# Train Decision Tree Classifer
clf = clf.fit(X_train, y_train)
# Predict the response for test dataset
y_pred = clf.predict(X_test)

### Question 11 : Évaluation du modèle

In [ ]:
print("Accuracy :", metrics.accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

**Interprétation :** *(à rédiger après exécution)*

### Question 12 : Visualisation de l'arbre

In [ ]:
dot_data = StringIO()
export_graphviz(clf, out_file=dot_data,
                filled=True, rounded=True,
                special_characters=True, feature_names=feature_cols, class_names=['0', '1'])
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
graph.write_png('diabetes.png')
Image(graph.create_png())

**a) Informations affichées dans chaque nœud**

- La **condition de séparation** (par exemple `glucose ≤ 127.5`) : la variable et le seuil choisis pour couper le nœud en deux (réponse vraie à gauche, fausse à droite). Les feuilles n'en ont pas.
- **gini** : l'impureté du nœud (0 = nœud pur, ne contenant qu'une seule classe ; 0,5 = mélange équilibré des deux classes).
- **samples** : le nombre d'observations d'apprentissage qui arrivent dans ce nœud.
- **value** : la répartition de ces observations par classe, `[nombre de 0, nombre de 1]`.
- **class** : la classe majoritaire du nœud, c'est-à-dire la classe prédite si l'on s'arrête là. La couleur du nœud correspond à cette classe, et elle est d'autant plus foncée que le nœud est pur.

**b) L'arbre est-il élagué ?**

**Non.** Avec les paramètres par défaut (pas de `max_depth`, ni de `min_samples_split`), l'arbre se développe jusqu'à ce que toutes les feuilles soient pures (gini = 0). Il est donc très profond et très ramifié, ce qui indique un risque de **surapprentissage** : il mémorise l'ensemble d'apprentissage au lieu de généraliser.

### Question 13 : Optimisation de l'arbre

Paramètres de `DecisionTreeClassifier()` :
- `criterion` : `"gini"` (par défaut) ou `"entropy"` (gain d'information) ;
- `splitter` : `"best"` (meilleure répartition) ou `"random"` (meilleure répartition parmi des choix aléatoires) ;
- `max_depth` : profondeur maximale de l'arbre (nommé `max_degree` dans le sujet, mais le paramètre de scikit-learn est bien `max_depth`). Une valeur trop grande provoque du surapprentissage, une valeur trop petite du sous-apprentissage.

**a) Entropie et profondeur maximale de 3**

In [ ]:
clf2 = DecisionTreeClassifier(criterion="entropy", max_depth=3)
clf2 = clf2.fit(X_train, y_train)
y_pred2 = clf2.predict(X_test)

print("Accuracy :", metrics.accuracy_score(y_test, y_pred2))
print(confusion_matrix(y_test, y_pred2))

**b) Nouvelle visualisation de l'arbre**

In [ ]:
dot_data = StringIO()
export_graphviz(clf2, out_file=dot_data,
                filled=True, rounded=True,
                special_characters=True, feature_names=feature_cols, class_names=['0', '1'])
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
graph.write_png('diabetes_optimise.png')
Image(graph.create_png())

**Constat :** *(à rédiger après exécution : taille de l'arbre, lisibilité, comparaison des accuracy avant et après)*

**c) Pré-élagage ou post-élagage ?**

C'est un **pré-élagage** : `max_depth=3` limite la croissance de l'arbre **pendant** sa construction, en l'arrêtant dès qu'il atteint la profondeur 3. Un post-élagage consisterait à construire d'abord l'arbre complet, puis à supprimer après coup les branches peu utiles (par exemple avec le paramètre `ccp_alpha`).